<a href="https://colab.research.google.com/github/ShamScripts/Employee-Attrition-RF-Model/blob/main/Random_Forest_Employee_Attrition_MLDemo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Employee attrition — ML demo (Random Forest focus)

**IBM HR Analytics** — end-to-end workflow: explore the data, compare **logistic regression**, a **decision tree**, and **random forest**, then interpret results with business context.

### Why this notebook

- **Baselines:** linear (logistic) vs nonlinear (tree) vs ensemble (forest) on the **same** train/test split.
- **Imbalanced target:** attrition is rare; we report precision, recall, F1, and **ROC-AUC**, not accuracy alone.
- **Actionable outputs:** feature importance, **predicted risk** (probability), and a simple **what-if** on overtime.

### How to run

From the project folder (with `ibm_hr_attrition.csv`), run all cells top to bottom. Requirements: `pandas`, `numpy`, `matplotlib`, `seaborn`, `scikit-learn`.

---

### Table of contents

| # | Section |
|---|---------|
| 1 | Setup and load data |
| 2 | EDA |
| 3 | Preprocessing |
| 4 | Models (LR, DT, RF) |
| 5 | Metrics and confusion matrices |
| 6 | ROC-AUC |
| 7 | Feature importance (RF) |
| 8 | Model comparison |
| 9 | Attrition risk score |
| 10 | What-if (overtime) |
| 11 | Business insights and conclusion |

## 1. Setup and load data

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    auc,
)

for _style in ("seaborn-v0_8-whitegrid", "seaborn-whitegrid", "ggplot"):
    try:
        plt.style.use(_style)
        break
    except OSError:
        continue

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 10

ROOT = Path.cwd()
if not (ROOT / "ibm_hr_attrition.csv").exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / "ibm_hr_attrition.csv"
PLOTS_DIR = ROOT / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "Attrition"
RANDOM_STATE = 42
COLORS = {"no": "#4C72B0", "yes": "#C44E52", "accent": "#55A868"}

print("Dataset path:", DATA_PATH.resolve())

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
print("Shape:", df_raw.shape)
df_raw.head()

## 2. Exploratory data analysis (EDA)

Quick checks: target balance and a few relationships that matter for HR storytelling.

In [ ]:
# Class balance
vc = df_raw[TARGET].value_counts()
prop = df_raw[TARGET].value_counts(normalize=True)
fig, ax = plt.subplots(figsize=(5.5, 4))
vc.plot(kind="bar", ax=ax, color=[COLORS["no"], COLORS["yes"]], edgecolor="white")
ax.set_title("Attrition class counts")
ax.set_xlabel("Attrition")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
print("Proportions:\n", prop.round(3))

In [ ]:
# Overtime vs attrition (if column exists)
if "OverTime" in df_raw.columns:
    ct = pd.crosstab(df_raw["OverTime"], df_raw[TARGET], normalize="index") * 100
    fig, ax = plt.subplots(figsize=(6, 4))
    ct.plot(kind="bar", ax=ax, color=[COLORS["no"], COLORS["yes"]], edgecolor="white")
    ax.set_title("Attrition rate (%) by OverTime")
    ax.set_ylabel("Percent within OverTime group")
    ax.legend(title=TARGET)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print("No OverTime column in dataset.")

## 3. Preprocessing

Drop ID-like or constant columns, encode categoricals with **one-hot** (`drop_first=True`), stratified **train/test split** (25% test), and keep the positive class rate stable across splits.

In [ ]:
df = df_raw.copy().dropna()
DROP = ["EmployeeCount", "EmployeeNumber", "Over18", "StandardHours"]
df = df.drop(columns=[c for c in DROP if c in df.columns], errors="ignore")

y = (df[TARGET].astype(str).str.strip() == "Yes").astype(int)
X = df.drop(columns=[TARGET])
X = pd.get_dummies(X, drop_first=True)

print("Encoded features:", X.shape[1])
print("Positive class rate (leave):", round(y.mean(), 4))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
print("Train:", X_train.shape[0], "| Test:", X_test.shape[0])
print("Train churn rate:", round(y_train.mean(), 4), "| Test:", round(y_test.mean(), 4))

## 4. Models

1. **Logistic regression** — linear baseline; `StandardScaler` inside a pipeline for numerical stability; `class_weight='balanced'` for the minority class.
2. **Decision tree** — single tree with depth/leaf limits to limit extreme overfitting while still showing train vs test gap.
3. **Random forest** — main model: `class_weight='balanced_subsample'`, many trees, parallel `n_jobs=-1`.

In [ ]:
model_lr = Pipeline([
    ("scaler", StandardScaler()),
    (
        "lr",
        LogisticRegression(
            max_iter=10000,
            solver="lbfgs",
            random_state=RANDOM_STATE,
            class_weight="balanced",
        ),
    ),
])
model_lr.fit(X_train, y_train)

model_dt = DecisionTreeClassifier(
    random_state=RANDOM_STATE,
    max_depth=8,
    min_samples_leaf=10,
    class_weight="balanced",
)
model_dt.fit(X_train, y_train)

model_rf = RandomForestClassifier(
    n_estimators=200,
    random_state=RANDOM_STATE,
    class_weight="balanced_subsample",
    n_jobs=-1,
    min_samples_leaf=2,
)
model_rf.fit(X_train, y_train)

models = {
    "Logistic Regression": model_lr,
    "Decision Tree": model_dt,
    "Random Forest": model_rf,
}
print("All models trained.")

## 5. Metrics and confusion matrices

On **imbalanced** attrition data, prioritize **recall** on the minority class (catching leavers) and **F1**; use accuracy only as a secondary check.

In [ ]:
def metrics_row(name, model):
    pred = model.predict(X_test)
    return {
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
    }


metrics_df = pd.DataFrame([metrics_row(n, m) for n, m in models.items()])
print(metrics_df.round(4).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, (name, model) in zip(axes, models.items()):
    pred = model.predict(X_test)
    cm = confusion_matrix(y_test, pred)
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    for (j, k), v in np.ndenumerate(cm):
        ax.text(k, j, int(v), ha="center", va="center", color="black")
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle("Confusion matrices (0=stay, 1=leave)", y=1.05)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "fig_confusion_matrices.png", bbox_inches="tight")
plt.show()

In [ ]:
# Classification report for Random Forest (main model)
print(classification_report(y_test, model_rf.predict(X_test), target_names=["Stay", "Leave"]))

## 6. ROC curves and AUC (test set)

ROC summarizes tradeoffs between true and false positives across thresholds; **AUC** is a single-number summary (higher is generally better for ranking risk).

In [ ]:
plt.figure(figsize=(7, 6))
for name, model in models.items():
    if hasattr(model, "predict_proba"):
        p1 = model.predict_proba(X_test)[:, 1]
    else:
        continue
    fpr, tpr, _ = roc_curve(y_test, p1)
    a = auc(fpr, tpr)
    plt.plot(fpr, tpr, lw=2, label=f"{name} (AUC = {a:.3f})")

plt.plot([0, 1], [0, 1], "k--", alpha=0.35)
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC curves — test set")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "fig_roc.png", bbox_inches="tight")
plt.show()

## 7. Feature importance (Random Forest)

Gini-based importance from sklearn: useful for **storytelling** about which inputs the forest used most, not for strict causal claims.

In [ ]:
imp = pd.Series(model_rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(12)
fig, ax = plt.subplots(figsize=(9, 5.5))
imp.sort_values().plot(kind="barh", color=COLORS["accent"], ax=ax, edgecolor="white")
ax.set_title("Top features — Random Forest (mean decrease in impurity)")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "fig_feature_importance.png", bbox_inches="tight")
plt.show()

## 8. Model comparison (test accuracy)

Bar chart for a quick **relative** comparison; prefer the table above for imbalanced-aware metrics.

In [ ]:
acc_test = {n: accuracy_score(y_test, m.predict(X_test)) for n, m in models.items()}
fig, ax = plt.subplots(figsize=(7, 4))
pd.Series(acc_test).plot(kind="bar", ax=ax, color=COLORS["accent"], edgecolor="white")
ax.set_ylabel("Test accuracy")
ax.set_title("Test accuracy by model")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "fig_model_compare.png", bbox_inches="tight")
plt.show()

## 9. Attrition risk score

Use **predicted probability** of class 1 (leave) as a risk score for ranking employees for follow-up (not as an automatic HR decision).

In [ ]:
risk = model_rf.predict_proba(X_test)[:, 1]
risk_df = pd.DataFrame({"actual_leave": y_test.values, "p_leave": risk})
print(risk_df.head(10))
print("\nHigh-risk examples (top 5 by p_leave):")
print(risk_df.nlargest(5, "p_leave"))

## 10. What-if: overtime indicator

If one-hot columns include overtime, flip the indicator for one test row and compare **P(leave)**.

In [ ]:
row = X_test.iloc[[0]].copy()
base_prob = model_rf.predict_proba(row)[0, 1]
ot_cols = [c for c in row.columns if "OverTime" in c or "Overtime" in c]
print("Baseline P(leave) for first test row:", round(base_prob, 4))
print("Overtime-related columns:", ot_cols)

if ot_cols:
    row_whatif = row.copy()
    for c in ot_cols:
        row_whatif[c] = 1.0 - row_whatif[c]
    new_prob = model_rf.predict_proba(row_whatif)[0, 1]
    print("After flipping overtime indicator(s):", round(new_prob, 4))
    print("Delta:", round(new_prob - base_prob, 4))
else:
    print("No overtime dummy columns found after encoding.")

## 11. Business insights and conclusion

**Takeaways**

- **Random forest** usually balances **recall** and **stability** versus a single tree; logistic regression gives a **linear** baseline.
- Drivers with high importance (e.g., overtime, income, tenure-related features) deserve **qualitative** follow-up with HR policy, not automatic actions.
- **Limitations:** no causal claims; data may reflect historical bias; consider **threshold tuning**, **SHAP**, and **fairness review** before production.

**Next steps (optional)**

- Hyperparameter search (e.g., `RandomizedSearchCV`), **SMOTE** or other imbalance strategies, **MLOps** monitoring.
